### Phase 3: Modelling and Analysis

In [1]:
# =====================================================
# 🧠 PHASE 3 — MODELLING
# =====================================================
# Can run after a kernel restart
# Loads features, benchmark, cleans, computes targets
# Runs rolling LightGBM regression CV

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import warnings
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

In [3]:
# ==========================================================
# ---------------------- CONFIG -----------------------------
# ==========================================================

PROJECT_ROOT = Path().resolve()

if (PROJECT_ROOT / "data").exists():
    BASE_PATH = PROJECT_ROOT
else:
    BASE_PATH = PROJECT_ROOT.parent

DATA_DIR = BASE_PATH / "data"
FEATURE_DIR = DATA_DIR / "features"
MODELS_DIR = DATA_DIR / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Feature directory: {FEATURE_DIR}")

📂 Feature directory: C:\Users\jayan\Study Materials\ml_alpha_portfolio\ml-alpha-portfolio-model\notebooks\data\features


In [4]:
# ==========================================================
# 🔹 STEP 1: LOAD PRICE FEATURE PANEL
# ==========================================================

print("📥 STEP 1: Loading feature CSVs...")

dfs = []
for fp in FEATURE_DIR.glob("*.csv"):
    df = pd.read_csv(fp)
    df.columns = df.columns.str.strip().str.lower()

    if "ticker" not in df.columns:
        df["ticker"] = fp.stem

    dfs.append(df)

panel = pd.concat(dfs, ignore_index=True)

panel["date"] = pd.to_datetime(panel["date"], errors="coerce")
panel = panel.dropna(subset=["date", "ticker"])
panel = panel.sort_values(["ticker", "date"])

print(f"✅ Panel loaded: {len(panel)} rows")

# =====================================================
# SAVE FULL 96-STOCK PANEL DATASET
# =====================================================

panel.to_csv(DATA_DIR / "full_96_stock_panel.csv", index=False)
print("💾 Saved full 96-stock panel dataset")

📥 STEP 1: Loading feature CSVs...
✅ Panel loaded: 131555 rows
💾 Saved full 96-stock panel dataset


In [5]:
# ==========================================================
# 🔹 STEP 2: CREATE TARGET
# ==========================================================

print("🎯 STEP 2: Creating forward return target...")

panel["fwd_ret"] = (
    panel.groupby("ticker")["close"].shift(-5) / panel["close"] - 1
)

panel["target"] = (panel["fwd_ret"] > 0).astype(int)
panel = panel.dropna(subset=["fwd_ret"])

print("✅ Target ready")

🎯 STEP 2: Creating forward return target...
✅ Target ready


In [6]:
# =====================================================
# STEP 3: FEATURE BLOCKS (NO NLP RE-COMPUTATION)
# =====================================================

print("📊 STEP 3: Building feature blocks...")

technical_features = [
    c for c in panel.columns
    if any(x in c for x in ["rsi", "macd", "momentum", "ema"])
]

price_structure_features = [
    c for c in panel.columns
    if any(x in c for x in ["vol_", "volume"])
]

concall_features = [
    c for c in panel.columns
    if "concall_" in c
]

base_features = technical_features + price_structure_features
hybrid_features = list(set(base_features + concall_features))

print(f"Technical: {len(technical_features)}")
print(f"Price Structure: {len(price_structure_features)}")
print(f"Concall: {len(concall_features)}")
print(f"Total Hybrid: {len(hybrid_features)}")


📊 STEP 3: Building feature blocks...
Technical: 13
Price Structure: 5
Concall: 14
Total Hybrid: 32


In [7]:
# =====================================================
# STEP 4: ROLLING CV WITH PROPER NORMALIZATION
# =====================================================

print("🔄 STEP 4: Running rolling CV...")

LGB_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True,
)


def rolling_train(panel, features, min_train_days=120, test_window=20):

    results = []
    dates = sorted(panel["date"].unique())

    for i in tqdm(range(min_train_days, len(dates) - test_window)):

        train = panel[panel["date"] <= dates[i]]
        test = panel[
            (panel["date"] > dates[i]) &
            (panel["date"] <= dates[i + test_window])
        ]

        if train.empty or test.empty:
            continue

        X_train = train[features]
        y_train = train["target"]

        X_test = test[features]
        y_test = test["target"]

        # ------------------------------
        # ✅ MEDIAN IMPUTATION
        # ------------------------------
        medians = X_train.median()
        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        # ------------------------------
        # ✅ NORMALIZATION (CORRECT PLACE)
        # Fit ONLY on train
        # ------------------------------
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = LGBMClassifier(**LGB_PARAMS)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="binary_logloss",
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        preds = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, preds)

        results.append({"auc": auc})

    return pd.DataFrame(results)

🔄 STEP 4: Running rolling CV...


In [8]:
# =====================================================
# STEP 5: MODEL COMPARISON
# =====================================================

print("🚀 STEP 5: Running models...")

cv_base = rolling_train(panel, base_features)
cv_base["model"] = "tech_price"

cv_hybrid = rolling_train(panel, hybrid_features)
cv_hybrid["model"] = "hybrid"

cv_compare = pd.concat([cv_base, cv_hybrid], ignore_index=True)

if not cv_compare.empty:

    auc_summary = cv_compare.groupby("model")["auc"].mean()

    print("\n📈 Average AUC by Model:")
    print(auc_summary)

    if "hybrid" in auc_summary.index:
        auc_diff = auc_summary["hybrid"] - auc_summary["tech_price"]
        print(f"\n🔍 AUC Difference (Hybrid - Base): {auc_diff:.6f}")

else:
    print("⚠️ No valid CV folds produced.")

print("\n✅ Phase 3 completed successfully.")

🚀 STEP 5: Running models...


100%|██████████| 1263/1263 [37:22<00:00,  1.78s/it]   


📈 Average AUC by Model:
model
hybrid        0.521052
tech_price    0.521973
Name: auc, dtype: float64

🔍 AUC Difference (Hybrid - Base): -0.000921

✅ Phase 3 completed successfully.


In [9]:
# =====================================================
# STEP 6A: CREATE MULTI-HORIZON TARGETS
# =====================================================

print("\n📐 STEP 6: Multi-Horizon Regression Setup...")

horizons = [1, 5, 10, 20]

for h in horizons:
    panel[f"fwd_ret_{h}"] = (
        panel.groupby("ticker")["close"].shift(-h) / panel["close"] - 1
    )

panel = panel.dropna(subset=[f"fwd_ret_{h}" for h in horizons])

print("✅ Forward returns created for horizons:", horizons)



📐 STEP 6: Multi-Horizon Regression Setup...
✅ Forward returns created for horizons: [1, 5, 10, 20]


In [10]:
# =====================================================
# STEP 6B: REGRESSION ROLLING CV
# =====================================================

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

LGB_REG_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)


def rolling_regression(panel, features, horizon):

    results = []
    dates = sorted(panel["date"].unique())

    for i in range(120, len(dates) - 20):

        train = panel[panel["date"] <= dates[i]]
        test = panel[
            (panel["date"] > dates[i]) &
            (panel["date"] <= dates[i + 20])
        ]

        if train.empty or test.empty:
            continue

        y_col = f"fwd_ret_{horizon}"

        X_train = train[features]
        y_train = train[y_col]

        X_test = test[features]
        y_test = test[y_col]

        medians = X_train.median()
        X_train = X_train.fillna(medians)
        X_test = X_test.fillna(medians)

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = LGBMRegressor(**LGB_REG_PARAMS)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)

        # Information Coefficient (rank correlation)
        ic, _ = spearmanr(y_test, preds)

        results.append({
            "rmse": rmse,
            "r2": r2,
            "ic": ic
        })

    return pd.DataFrame(results)


# Run regression for hybrid model across horizons

regression_results = []

for h in horizons:
    print(f"🔄 Running regression for horizon {h} days")
    df_res = rolling_regression(panel, hybrid_features, h)
    df_res["horizon"] = h
    regression_results.append(df_res)

regression_results = pd.concat(regression_results)

print("\n📊 Regression Summary:")
print(regression_results.groupby("horizon").mean())

🔄 Running regression for horizon 1 days
🔄 Running regression for horizon 5 days
🔄 Running regression for horizon 10 days
🔄 Running regression for horizon 20 days

📊 Regression Summary:
             rmse        r2        ic
horizon                              
1        0.022271 -0.053833  0.010557
5        0.051723 -0.127720  0.033480
10       0.073232 -0.179775  0.036698
20       0.104733 -0.262033  0.067551


In [11]:
# =====================================================
# STEP 7A: FETCH NSE SECTOR DATA
# =====================================================

import requests

print("\n🏢 STEP 7: Fetching NSE sector data...")

url = "https://www.nseindia.com/api/equity-stockIndices?index=NIFTY%20500"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
data = response.json()

sector_map = {}

for stock in data["data"]:
    symbol = stock["symbol"]
    sector = stock.get("industry", "Unknown")
    sector_map[symbol] = sector

panel["sector"] = panel["ticker"].map(sector_map)

print("✅ Sector mapping complete")


🏢 STEP 7: Fetching NSE sector data...
✅ Sector mapping complete


In [12]:
# =====================================================
# STEP 7B: SECTOR-WISE CLASSIFICATION PERFORMANCE
# =====================================================

print("\n📊 Sector-wise AUC dispersion...")

sector_auc = []

for sector in panel["sector"].dropna().unique():

    sector_data = panel[panel["sector"] == sector]

    if len(sector_data) < 200:
        continue

    cv = rolling_train(sector_data, hybrid_features)

    if not cv.empty:
        sector_auc.append({
            "sector": sector,
            "mean_auc": cv["auc"].mean()
        })

sector_auc_df = pd.DataFrame(sector_auc)

print(sector_auc_df.sort_values("mean_auc", ascending=False))


📊 Sector-wise AUC dispersion...


100%|██████████| 1243/1243 [09:01<00:00,  2.30it/s]

    sector  mean_auc
0  Unknown  0.520544


In [13]:
# ANOVA test
from scipy.stats import f_oneway

sector_groups = []

for sector in sector_auc_df["sector"]:
    auc_values = rolling_train(
        panel[panel["sector"] == sector],
        hybrid_features
    )["auc"]

    sector_groups.append(auc_values)

f_stat, p_val = f_oneway(*sector_groups)

print("\n📊 ANOVA Test Across Sectors")
print("F-stat:", f_stat)
print("p-value:", p_val)

100%|██████████| 1243/1243 [08:57<00:00,  2.31it/s]


TypeError: at least two inputs are required; got 1.

In [ ]:
# =====================================================
# STEP 8A: TRAIN FINAL HYBRID MODEL ON FULL DATA
# =====================================================

print("\n📊 STEP 8: SHAP Analysis...")

X = panel[hybrid_features]
y = panel["target"]

medians = X.median()
X = X.fillna(medians)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LGBMClassifier(**LGB_PARAMS)
model.fit(X_scaled, y)

# =====================================================
# STEP 8B: SHAP EXPLANATION
# =====================================================

import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

# Global importance
shap.summary_plot(shap_values[1], X, plot_type="bar")

# Feature distribution importance
shap.summary_plot(shap_values[1], X)